In [1]:
import pandas as pd

df = pd.read_csv("moto_imu_sim/imu_data.csv")

In [2]:
W, S = 400, 200
rows = []
for rid, g in df.groupby("ride_id"):
    for start in range(0, len(g) - W + 1, S):
        rows.append((rid, start, g["label"].iloc[0]))
win = pd.DataFrame(rows, columns = ["ride_id", "start", "label"])
print(len(win))
print(win.groupby("label").size())


10714
label
aggressive    2379
calm          3573
normal        4762
dtype: int64


here, g is the dataframe (table) seperated by "group_by("ride_id"), and rid takes the ride_id
. Actually the structure of g is (ride_id, dataframe) so rid takes ride_id and g takes the dataframe table

In inner loop the start is 0, leng(g) - W + 1 is the end and S is the step size , it determinest the window size 


row is the list of tuples like
rows = [
    ("R001", 0, "calm"),
    ("R001, 200, "calm"),
    ....
]


iloc[0] is to get the label of the first entry of g.


In [3]:
g = df[df["ride_id"] == "R002"]
w = g.iloc[0:400]
w2 = g.iloc[200:600]
print(len(w), w["gx"].std(), w2["gx"].std())

400 1.4200467203800766 1.8149350549829777


In [4]:
g = df[df["ride_id"] == "R002"]
print(g["label"].iloc[0], "ride std:", g["gx"].std())
for s in [0, 10000, 20000, 30000, 40000, 50000]:
    print(s, g["gx"].iloc[s:s+400].std())

aggressive ride std: 12.90683430689985
0 1.4200467203800766
10000 2.876189560542244
20000 2.2023830410263554
30000 7.4717691967059725
40000 0.9729728124365916
50000 1.3635936808700537


In [5]:
vals = []
for s in range(0, len(g) - 400 + 1,200):
    vals.append(g["gx"].iloc[s:s+400].std())
print(pd.Series(vals).describe()) 

count    297.000000
mean       7.781920
std        7.848828
min        0.933178
25%        2.692707
50%        3.675836
75%       11.408469
max       35.475854
dtype: float64


In [6]:
g = df[df["ride_id"] == "R001"]
vals = []
for s in range(0, len(g) - 400 + 1, 200):
    vals.append(g["gx"].iloc[s:s+400].std())
print(pd.Series(vals).describe())

count    298.000000
mean       3.145349
std        0.982318
min        1.544519
25%        2.466783
50%        2.940678
75%        3.477655
max        7.373017
dtype: float64


In [ ]:
rows = []
for rid, g in df.groupby("ride_id"):
    for start in range(0, len(g) - W + 1, S):
        w = g.iloc[start:start+W]
        rows.append((rid, start, g["label"].iloc[0], w["gx"].std()))
##print(rows)
win = pd.DataFrame(rows, columns=["ride_id", "start", "label", "gx_std"])
print(win.groupby("label")["gx_std"].describe())

[('R001', 0, 'calm', np.float64(2.9083418627910307)), ('R001', 200, 'calm', np.float64(3.435697940349644)), ('R001', 400, 'calm', np.float64(3.156414478743712)), ('R001', 600, 'calm', np.float64(2.0979615721635687)), ('R001', 800, 'calm', np.float64(2.907538504829216)), ('R001', 1000, 'calm', np.float64(3.07589836617068)), ('R001', 1200, 'calm', np.float64(2.2285310633243562)), ('R001', 1400, 'calm', np.float64(2.2770077324066573)), ('R001', 1600, 'calm', np.float64(2.523316813207514)), ('R001', 1800, 'calm', np.float64(2.28053366940492)), ('R001', 2000, 'calm', np.float64(1.9400641749783)), ('R001', 2200, 'calm', np.float64(1.7559520834939377)), ('R001', 2400, 'calm', np.float64(1.9827848217681894)), ('R001', 2600, 'calm', np.float64(2.7342109666108456)), ('R001', 2800, 'calm', np.float64(3.349148733249413)), ('R001', 3000, 'calm', np.float64(2.7177441000255262)), ('R001', 3200, 'calm', np.float64(2.7396695234572497)), ('R001', 3400, 'calm', np.float64(2.203018766204852)), ('R001', 36

here we are just evaluating the gx_std over all the windows and finding their attirbutes; which is not giving a full detail i.e window gx_std is a weak feature on it's own.  median of all three labels are kinda similar, on 75th percentile, only agressive is seperable and others are not distinguishable. now we add the other axis for more information "ax_std" for forward and backward pushes

For this we edit the loop which we already have.


In [12]:
rows = []
for rid, g in df.groupby("ride_id"):
    for start in range(0, len(g) - W + 1, S):
        w = g.iloc[start:start + W]
        rows.append((rid, start, g["label"].iloc[0], w["gx"].std(), w["gx"].abs().max(), w["ax"].std()))

win = pd.DataFrame(rows, columns=["ride_id",  "start", "label", "gx_std", "gx_peak", "ax_std"])
print(win.groupby("label")[["gx_std", "gx_peak", "ax_std"]].median())
print(win.groupby("label")["ax_std"].describe())

              gx_std    gx_peak    ax_std
label                                    
aggressive  3.658884  11.352539  0.133853
calm        3.278262  10.131836  0.054266
normal      3.448111  10.528564  0.079292
             count      mean       std       min       25%       50%  \
label                                                                  
aggressive  2379.0  0.159134  0.102876  0.010525  0.087996  0.133853   
calm        3573.0  0.066401  0.043664  0.010675  0.036131  0.054266   
normal      4762.0  0.095552  0.063802  0.009024  0.049421  0.079292   

                 75%       max  
label                           
aggressive  0.201901  0.671688  
calm        0.080951  0.314933  
normal      0.127391  0.476699  


here, the typical range to acess a label or any data is 25th to 75th percentile : so for calm, 0.036 to 0.081 and max is 0.31 and so on
. If we look closely this range for calm and agressive are seperable ie. ----------25-75 of calm-------25-75 of agressive---------
the normal overlaps with both
